In [1]:
# ==== Model Builder (Option A: intersection join, t+1 target) ====
# Dependencies
import warnings, math, os
from pathlib import Path
import pandas as pd, numpy as np

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge, Lasso, LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import joblib

# ---- paths ----

PROC = Path("./processed")
PROC.mkdir(exist_ok=True)
OUT = PROC

warnings.filterwarnings("ignore")

# ---- helpers ----
META_COLS = {"quarter_end","year","quarter","quarter_label"}

def _pick_value_col(df: pd.DataFrame) -> str:
    """
    From a standardized processed CSV (with metadata columns present),
    return the single *numeric* series column to use as the value.
    """
    candidates = [c for c in df.columns if c not in META_COLS]
    # Prefer the one that is most numeric
    best, best_non_na = None, -1
    for c in candidates:
        num = pd.to_numeric(df[c], errors="coerce")
        non_na = int(num.notna().sum())
        if non_na > best_non_na:
            best, best_non_na = c, non_na
    if best is None:
        raise ValueError("Could not find a numeric value column.")
    return best

def load_series(proc_filename: str, rename_to: str) -> pd.Series:
    """Load a processed CSV and return a numeric Series indexed by quarter_end."""
    df = pd.read_csv(PROC / proc_filename, parse_dates=["quarter_end"])
    vcol = _pick_value_col(df)
    ser = (pd.to_numeric(df[vcol], errors="coerce")
             .rename(rename_to)
             .set_axis(df["quarter_end"]))
    # If duplicates exist per quarter_end (shouldn't), take last
    ser = ser.groupby(ser.index).last().sort_index()
    return ser

def rmspe(y, yhat):
    mask = y != 0
    if mask.any():
        return math.sqrt(np.mean(((yhat[mask] - y[mask]) / y[mask])**2)) * 100
    return np.nan

def print_metrics(y_true, y_pred, label=""):
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_pred - y_true) / np.where(y_true==0, np.nan, y_true))) * 100
    print(f"{label}RMSE={rmse:6.3f} | MAE={mae:6.3f} | R2={r2:6.3f} | MAPE={mape:6.2f}%")

# ---- load all series ----
series_map = {
    "quarterly_excess_ret.csv" : "excess_ret",
    "quarterly_cpi_yoy.csv"    : "cpi_yoy",
    "quarterly_gdp_yoy.csv"    : "gdp_yoy",
    "quarterly_repo_level.csv" : "repo",
    "quarterly_repo_chg_bps.csv":"repo_chg_bps",
    "quarterly_rain_anom.csv"  : "rain_anom",
}

loaded = {}
missing_files = []
for fn, name in series_map.items():
    path = PROC / fn
    if not path.exists():
        missing_files.append(fn)
        continue
    loaded[name] = load_series(fn, name)

if missing_files:
    print("⚠ Missing files (skipped):", missing_files)

if not loaded:
    raise RuntimeError("No series loaded. Check PROC path & filenames.")

# ---- inner-join on common quarters ----
df = pd.concat(loaded.values(), axis=1, join="inner").sort_index()
df.index.name = "quarter_end"

# quick QA
print("Data coverage:", df.index.min().date(), "→", df.index.max().date(), "| rows:", df.shape[0])
print(df.tail(3))

# ---- target: next-quarter excess return ----
# We predict T+1 excess_ret using features at time T (no leakage).
df["target_excess_ret_next"] = df["excess_ret"].shift(-1)

# ---- simple feature engineering knobs (easy to tweak)
# add modest lags to economic features to reflect reporting delays.
LAGS = {
    # 'cpi_yoy': 1,
    # 'gdp_yoy': 1,
    # 'repo': 0,
    # 'repo_chg_bps': 0,
    # 'rain_anom': 0,
}
for col, lag in LAGS.items():
    if col in df.columns and lag:
        df[f"{col}_lag{lag}"] = df[col].shift(lag)

# Quarter-of-year encodings (cyclical)
per = df.index.to_period("Q")
q_num = per.quarter
df["q_sin"] = np.sin(2*np.pi*(q_num/4.0))
df["q_cos"] = np.cos(2*np.pi*(q_num/4.0))

# Momentum of the spread 
df["excess_ret_lag1"] = df["excess_ret"].shift(1)
df["excess_ret_ma4"]  = df["excess_ret"].rolling(4).mean()

# Lagged macro features and change rates
for col in ["cpi_yoy", "gdp_yoy", "repo_chg_bps", "rain_anom"]:
    if col in df.columns:
        df[f"{col}_lag1"] = df[col].shift(1)

for col in ["cpi_yoy", "gdp_yoy", "rain_anom"]:
    if col in df.columns and f"{col}_lag1" in df.columns:
        df[f"{col}_diff"] = df[col] - df[f"{col}_lag1"]

# ---- finalize features/labels ----
feature_cols = [
    "cpi_yoy_lag1", "gdp_yoy_lag1", "repo_chg_bps_lag1", "rain_anom_lag1",
    "cpi_yoy_diff", "gdp_yoy_diff", "rain_anom_diff",
    "excess_ret_lag1", "excess_ret_ma4", "q_sin", "q_cos",
]
feature_cols = [c for c in feature_cols if c in df.columns]

# Re-create X,y cleanly
X = df[feature_cols]
y = df["target_excess_ret_next"]
data = pd.concat([X, y], axis=1).dropna()
X = data[feature_cols]
y = data["target_excess_ret_next"]

print("Final features:", feature_cols)
print("Rows after cleanup:", len(X))

# ---- time-series CV setup ----
# Use expanding-window folds; last fold acts like a recent out-of-sample.
n_splits = 5 if len(data) >= 40 else max(3, len(data)//10)  # heuristic to avoid tiny folds
tscv = TimeSeriesSplit(n_splits=n_splits)

# ---- candidate models ----
elastic_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(random_state=42, max_iter=10000)),
])
elastic_grid = {
    "model__alpha": [0.01, 0.05, 0.1, 0.2],
    "model__l1_ratio": [0.2, 0.5, 0.8],
}

candidates = {
    "ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=3.0, random_state=42)),
    ]),
    "lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=0.01, random_state=42, max_iter=5000)),
    ]),
    "gbr": Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("model", GradientBoostingRegressor(random_state=42,
                                            n_estimators=600,
                                            learning_rate=0.03,
                                            max_depth=3,
                                            subsample=0.9)),
    ]),
}

candidates["elasticnet"] = GridSearchCV(elastic_pipe, elastic_grid,
                                       cv=tscv, scoring="neg_root_mean_squared_error",
                                       n_jobs=1, refit=True)

# ---- backtest & model selection ----
cv_rows = []
scores = []
for name, pipe in candidates.items():
    y_pred_full = pd.Series(index=y.index, dtype=float)
    fold_no = 0
    for train_idx, test_idx in tscv.split(X):
        fold_no += 1
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        pipe.fit(X_tr, y_tr)
        y_hat = pd.Series(pipe.predict(X_te), index=y_te.index)

        # store per-fold predictions for later inspection
        y_pred_full.loc[y_te.index] = y_hat

        # metrics
        rmse = mean_squared_error(y_te, y_hat, squared=False)
        mae  = mean_absolute_error(y_te, y_hat)
        r2   = r2_score(y_te, y_hat)
        cv_rows.append({
            "model": name, "fold": fold_no, "start": X_te.index.min(), "end": X_te.index.max(),
            "rmse": rmse, "mae": mae, "r2": r2
        })

    # summarize per model
    valid_mask = y_pred_full.notna()
    rmse = mean_squared_error(y[valid_mask], y_pred_full[valid_mask], squared=False)
    mae  = mean_absolute_error(y[valid_mask], y_pred_full[valid_mask])
    r2   = r2_score(y[valid_mask], y_pred_full[valid_mask])
    scores.append((name, rmse, mae, r2))
    print_metrics(y[valid_mask], y_pred_full[valid_mask], label=f"[{name}] ")

cv_df = pd.DataFrame(cv_rows).sort_values(["model","fold"])
cv_df.to_csv(OUT/"cv_folds_metrics.csv", index=False)

scores_df = pd.DataFrame(scores, columns=["model","rmse","mae","r2"]).sort_values("rmse")
print("\n=== CV Summary (lower RMSE is better) ===")
print(scores_df)

best_name = scores_df.iloc[0]["model"]
best_pipe = candidates[best_name]
print(f"\nSelected model: {best_name}")

# ---- refit best on ALL data & save artifacts ----
best_pipe.fit(X, y)
joblib.dump({
    "pipeline": best_pipe,
    "feature_cols": feature_cols,
    "index": X.index,
    "meta": {
        "target": "target_excess_ret_next",
        "option": "A_inner_join_nextQ",
        "n_splits": n_splits
    }
}, OUT/"best_model.joblib")

# ---- full in-sample fitted vs actual & last-available forecast ----
fitted = pd.Series(best_pipe.predict(X), index=y.index, name="y_hat")
fit_df = pd.DataFrame({"y_true": y, "y_hat": fitted})
fit_df.to_csv(OUT/"fitted_vs_actual.csv")

# if you want a “true” next-quarter forecast using the most recent row:
latest_x = X.iloc[[-1]]
latest_quarter = latest_x.index[0]
nextq_pred = float(best_pipe.predict(latest_x))
print(f"\nLatest available quarter = {latest_quarter.date()}  →  predicted next-Q excess_ret = {nextq_pred:.4f}")
with open(OUT/"latest_forecast.txt","w") as f:
    f.write(f"{latest_quarter.date()},{nextq_pred:.6f}\n")

# ---- convenience: export the final training frame ----
data.to_csv(OUT/"model_dataset.csv", index=True)  # index = quarter_end
print(f"\nArtifacts written to: {OUT.resolve()}")


Data coverage: 2014-03-31 → 2024-09-30 | rows: 43
             excess_ret   cpi_yoy    gdp_yoy  repo  repo_chg_bps  rain_anom
quarter_end                                                                
2024-03-31     0.013611  5.014307   6.366486   6.5           0.0  -5.906736
2024-06-30     0.083945  4.904469   7.384530   6.5           0.0  -5.906736
2024-09-30     0.004268  4.244829  15.613499   6.5           0.0   7.564767
Final features: ['cpi_yoy_lag1', 'gdp_yoy_lag1', 'repo_chg_bps_lag1', 'rain_anom_lag1', 'cpi_yoy_diff', 'gdp_yoy_diff', 'rain_anom_diff', 'excess_ret_lag1', 'excess_ret_ma4', 'q_sin', 'q_cos']
Rows after cleanup: 39
[ridge] RMSE= 0.102 | MAE= 0.073 | R2=-2.287 | MAPE=360.27%
[lasso] RMSE= 0.081 | MAE= 0.063 | R2=-1.075 | MAPE=244.92%
[gbr] RMSE= 0.077 | MAE= 0.063 | R2=-0.912 | MAPE=225.40%
[elasticnet] RMSE= 0.063 | MAE= 0.053 | R2=-0.279 | MAPE=119.74%

=== CV Summary (lower RMSE is better) ===
        model      rmse       mae        r2
3  elasticnet  0.063307 

In [2]:
# --- Build CV benchmarks (models + baselines) and save tidy tables
def eval_cv(estimator, X, y, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    preds = pd.Series(index=y.index, dtype=float); rows=[]
    for k,(tr,te) in enumerate(tscv.split(X),1):
        Xtr,Xte,ytr,yte = X.iloc[tr],X.iloc[te],y.iloc[tr],y.iloc[te]
        estimator.fit(Xtr,ytr)
        p = pd.Series(estimator.predict(Xte), index=yte.index)
        preds.loc[yte.index] = p
        rows.append({"fold":k,
                     "start":Xte.index.min().date(),"end":Xte.index.max().date(),
                     "rmse":mean_squared_error(yte,p,squared=False),
                     "mae":mean_absolute_error(yte,p),
                     "r2":r2_score(yte,p)})
    valid = preds.notna()
    summary = {
        "rmse":mean_squared_error(y[valid],preds[valid],squared=False),
        "mae":mean_absolute_error(y[valid],preds[valid]),
        "r2":r2_score(y[valid],preds[valid]),
    }
    return pd.DataFrame(rows), pd.Series(summary), preds

def baseline_mean(X, y, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    preds = pd.Series(index=y.index, dtype=float); rows=[]
    for k,(tr,te) in enumerate(tscv.split(X),1):
        ytr,yte = y.iloc[tr], y.iloc[te]
        mu = ytr.mean()
        p = pd.Series(mu, index=yte.index)
        preds.loc[yte.index] = p
        rows.append({"fold":k,
                     "start":X.index[te].min().date(),"end":X.index[te].max().date(),
                     "rmse":mean_squared_error(yte,p,squared=False),
                     "mae":mean_absolute_error(yte,p),
                     "r2":r2_score(yte,p)})
    valid = preds.notna()
    summary = {
        "rmse":mean_squared_error(y[valid],preds[valid],squared=False),
        "mae":mean_absolute_error(y[valid],preds[valid]),
        "r2":r2_score(y[valid],preds[valid]),
    }
    return pd.DataFrame(rows), pd.Series(summary), preds

n_splits = 5 if len(X) >= 40 else max(3, len(X)//10)

# Baselines
bmean_folds, bmean_sum, bmean_pred = baseline_mean(X, y, n_splits)
blag_folds, blag_sum, blag_pred   = eval_cv(LinearRegression(), X[["excess_ret_lag1"]], y, n_splits)

baseline_df = pd.DataFrame([
    {"model":"baseline_mean", **bmean_sum.to_dict()},
    {"model":"baseline_lag1", **blag_sum.to_dict()},
])
print("\n=== Baseline comparison ===")
print(baseline_df)

# Reuse your 'candidates' dict from above
rows, summary = [], []
preds = {"baseline_mean": bmean_pred, "baseline_lag1": blag_pred}

for name, est in candidates.items():
    f, s, p = eval_cv(est, X, y, n_splits)
    f.insert(0,"model",name); rows.append(f)
    summary.append({"model":name, **s.to_dict()})
    preds[name] = p

cv_folds_df = pd.concat(rows, ignore_index=True)
cv_summary_df = pd.concat([
    pd.DataFrame(summary),
    pd.DataFrame([{"model":"baseline_mean", **bmean_sum.to_dict()}]),
    pd.DataFrame([{"model":"baseline_lag1", **blag_sum.to_dict()}]),
], ignore_index=True).sort_values("rmse")

pd.DataFrame({"y_true":y, **preds}).to_csv(OUT/"cv_predictions_w_baselines.csv")
cv_summary_df.to_csv(OUT/"cv_summary_models_vs_baselines.csv", index=False)

cv_folds_df.to_csv(OUT/"cv_folds_metrics_detailed.csv", index=False)

print(" -", OUT/"cv_predictions_w_baselines.csv")

print("Saved:")
print(" -", OUT/"cv_folds_metrics_detailed.csv")
print(" -", OUT/"cv_summary_models_vs_baselines.csv")


=== Baseline comparison ===
           model      rmse       mae        r2
0  baseline_mean  0.063499  0.052867 -0.286422
1  baseline_lag1  0.066159  0.054200 -0.396472
 - processed\cv_predictions_w_baselines.csv
Saved:
 - processed\cv_folds_metrics_detailed.csv
 - processed\cv_summary_models_vs_baselines.csv


Hold-out permutation importance (interpretability)

In [3]:
from sklearn.inspection import permutation_importance

best_row = cv_summary_df[~cv_summary_df["model"].isin(["baseline_mean","baseline_lag1"])].iloc[0]
best_name = best_row["model"]
best_est  = candidates[best_name]

h = 6 if len(X) > 10 else max(2, len(X)//5)  # ~1.5 years as holdout
X_tr, y_tr = X.iloc[:-h], y.iloc[:-h]
X_te, y_te = X.iloc[-h:],  y.iloc[-h:]

best_est.fit(X_tr, y_tr)
perm = permutation_importance(best_est, X_te, y_te,
                              n_repeats=200, random_state=42,
                              scoring="neg_root_mean_squared_error")
imp_df = (pd.DataFrame({"feature":X.columns, "importance":perm.importances_mean})
            .sort_values("importance", ascending=False))
imp_df.to_csv(OUT/"permutation_importance_holdout.csv", index=False)
imp_df.head(10)

,feature,importance
0,cpi_yoy_lag1,0.0
1,gdp_yoy_lag1,0.0
2,repo_chg_bps_lag1,0.0
3,rain_anom_lag1,0.0
4,cpi_yoy_diff,0.0
5,gdp_yoy_diff,0.0
6,rain_anom_diff,0.0
7,excess_ret_lag1,0.0
8,excess_ret_ma4,0.0
9,q_sin,0.0


Narrative: if excess_ret_lag1 dominates and macros are small/unstable → “momentum matters; macro adds little stable lift.”

EDA correlation matrix

In [4]:
corr = pd.concat([X, y.rename("target_next")], axis=1).corr(numeric_only=True)
corr.to_csv(OUT/"corr_matrix_for_slide.csv")
corr.round(2)

,cpi_yoy_lag1,gdp_yoy_lag1,repo_chg_bps_lag1,rain_anom_lag1,cpi_yoy_diff,gdp_yoy_diff,rain_anom_diff,excess_ret_lag1,excess_ret_ma4,q_sin,q_cos,target_next
cpi_yoy_lag1,1.00,0.26,0.33,-0.21,-0.42,0.14,0.34,0.18,0.30,-0.02,0.03,0.39
gdp_yoy_lag1,0.26,1.00,0.30,0.17,-0.39,-0.55,-0.06,0.30,0.45,-0.02,0.02,0.20
repo_chg_bps_lag1,0.33,0.30,1.00,0.14,-0.31,-0.04,-0.11,0.08,0.01,0.02,0.16,0.02
rain_anom_lag1,-0.21,0.17,0.14,1.00,-0.19,-0.04,-0.38,0.21,0.18,-0.02,0.00,0.25
cpi_yoy_diff,-0.42,-0.39,-0.31,-0.19,1.00,0.17,-0.15,-0.04,-0.17,0.05,-0.07,-0.21
gdp_yoy_diff,0.14,-0.55,-0.04,-0.04,0.17,1.00,0.11,0.06,-0.01,0.00,-0.05,-0.11
rain_anom_diff,0.34,-0.06,-0.11,-0.38,-0.15,0.11,1.00,0.03,0.05,-0.00,-0.00,-0.27
excess_ret_lag1,0.18,0.30,0.08,0.21,-0.04,0.06,0.03,1.00,0.53,0.09,0.16,0.01
excess_ret_ma4,0.30,0.45,0.01,0.18,-0.17,-0.01,0.05,0.53,1.00,0.04,0.02,0.21
q_sin,-0.02,-0.02,0.02,-0.02,0.05,0.00,-0.00,0.09,0.04,1.00,-0.00,-0.01


Simple ablation (Momentum-only vs Momentum+Macro)

In [5]:
def run_cv_rmse(est, X, y, n_splits):
    _, s, _ = eval_cv(est, X, y, n_splits)
    return s["rmse"]

mom_cols   = ["excess_ret_lag1","excess_ret_ma4","q_sin","q_cos"]
macro_cols = [c for c in feature_cols if c not in mom_cols]

est = Pipeline([("scaler",StandardScaler()),
                ("model",RandomForestRegressor(random_state=42,n_estimators=500,min_samples_leaf=2))])

rmse_mom   = run_cv_rmse(est, X[mom_cols], y, n_splits)
rmse_full  = run_cv_rmse(est, X[mom_cols + macro_cols], y, n_splits)

ablation = pd.DataFrame({
    "spec":["Momentum only","Momentum + Macro"],
    "rmse":[rmse_mom, rmse_full]
})
ablation.to_csv(OUT/"ablation_momentum_vs_macro.csv", index=False)
ablation


,spec,rmse
0,Momentum only,0.081112
1,Momentum + Macro,0.073017


In [6]:
with pd.ExcelWriter(OUT/"results_for_slides.xlsx") as xw:
    cv_summary_df.to_excel(xw, "cv_summary", index=False)
    cv_folds_df.to_excel(xw, "cv_folds_detailed", index=False)
    imp_df.to_excel(xw, "perm_importance_holdout", index=False)
    corr.to_excel(xw, "corr_matrix")
    ablation.to_excel(xw, "ablation", index=False)
print("Wrote", OUT/"results_for_slides.xlsx")

Wrote processed\results_for_slides.xlsx


In [7]:
# =========================================
# Cell 1 — Imports & paths
# =========================================
import os, sys, textwrap, json, warnings, math
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Try optional libs (nice-to-haves; code still works without them)
try:
    import sklearn
    from sklearn.metrics import (
        roc_curve, auc, precision_recall_curve, average_precision_score,
        confusion_matrix, r2_score, mean_squared_error, mean_absolute_error
    )
except Exception:
    sklearn = None

try:
    import scipy
    from scipy import stats
except Exception:
    scipy = None

# dataframe_image can export styled DataFrames to PNG; optional
try:
    import dataframe_image as dfi
except Exception:
    dfi = None

# Where to search for your uploaded files
SEARCH_DIRS = [Path.cwd(), Path("./processed/"), Path("./models/")]

# Output folders
OUT_ROOT   = Path("report_assets")
OUT_TABLES = OUT_ROOT / "tables"
OUT_CHARTS = OUT_ROOT / "charts"
for p in [OUT_ROOT, OUT_TABLES, OUT_CHARTS]:
    p.mkdir(parents=True, exist_ok=True)

# Friendly figure defaults (matplotlib only; no seaborn)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["figure.figsize"] = (7, 4.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

def find_file(candidates):
    """
    Return the first existing path that matches any candidate filename in SEARCH_DIRS.
    'candidates' can be a str or a list of possible filenames.
    """
    if isinstance(candidates, str):
        candidates = [candidates]
    for name in candidates:
        for base in SEARCH_DIRS:
            p = base / name
            if p.exists():
                return p
    return None

def save_table(df: pd.DataFrame, filename_stem: str, index: bool = False, float_fmt="{:,.3f}"):
    """
    Save a DataFrame to multiple formats:
      - CSV (always)
      - PNG (if 'dataframe_image' is available, else a Matplotlib fallback)
      - HTML (always)
    """
    csv_path  = OUT_TABLES / f"{filename_stem}.csv"
    html_path = OUT_TABLES / f"{filename_stem}.html"
    png_path  = OUT_TABLES / f"{filename_stem}.png"

    # CSV + HTML
    df.to_csv(csv_path, index=index)
    df.to_html(html_path, index=index, float_format=lambda x: float_fmt.format(x) if isinstance(x,(int,float,np.floating)) else x)

    # PNG export
    if dfi is not None:
        # Styled PNG via dataframe_image (best quality)
        styled = df.style.format(float_fmt)
        dfi.export(styled, png_path, table_conversion="matplotlib")
    else:
        # Simple Matplotlib fallback
        fig, ax = plt.subplots(figsize=(min(12, 0.8*(len(df.columns)+2)), 0.6*(len(df)+2)))
        ax.axis("off")
        tbl = ax.table(cellText=df.round(3).values, colLabels=df.columns, loc="center")
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(9)
        tbl.scale(1, 1.3)
        plt.tight_layout()
        fig.savefig(png_path, bbox_inches="tight")
        plt.close(fig)

    return {"csv": str(csv_path), "html": str(html_path), "png": str(png_path)}

def save_fig(fig, filename_stem: str):
    """Save a Matplotlib figure as PNG and SVG."""
    png = OUT_CHARTS / f"{filename_stem}.png"
    svg = OUT_CHARTS / f"{filename_stem}.svg"
    fig.tight_layout()
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    plt.close(fig)
    return {"png": str(png), "svg": str(svg)}

def sniff_binary_col(s: pd.Series):
    """Return True if the series looks like a binary label {0,1}/{False,True}/two-unique-values."""
    vals = pd.Series(s.dropna().unique())
    if len(vals) == 2:
        # normalize
        lowered = set(str(v).strip().lower() for v in vals)
        return True, lowered
    # {0,1} numeric?
    if set(pd.unique(s.dropna().astype(float).round().clip(0,1))) <= {0,1}:
        return True, {0,1}
    return False, set()

def pick_label_and_pred_cols(df: pd.DataFrame):
    """
    Heuristic: find label and prediction columns in OOF predictions.
      label candidates: y, y_true, target, label, outcome, actual
      proba/score candidates: y_pred_proba, proba, score, pred_proba, probability
      raw pred candidates: y_pred, prediction, pred
    """
    cols = {c.lower(): c for c in df.columns}
    label_cands = ["y", "y_true", "target", "label", "outcome", "actual"]
    proba_cands = ["y_pred_proba", "proba", "score", "pred_proba", "probability", "p_pred", "y_score"]
    pred_cands  = ["y_pred", "prediction", "pred", "predicted"]

    y_col = next((cols[k] for k in label_cands if k in cols), None)
    proba_col = next((cols[k] for k in proba_cands if k in cols), None)
    pred_col  = next((cols[k] for k in pred_cands  if k in cols), None)

    # fallbacks
    if y_col is None:
        # try any column that looks binary with common names
        for c in df.columns:
            is_bin, _ = sniff_binary_col(df[c])
            if is_bin:
                y_col = c
                break
    return y_col, proba_col, pred_col

def topn_numeric_columns(df, exclude=None, n=8):
    exclude = set(exclude or [])
    num_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
    # keep top by non-null count
    num_cols = sorted(num_cols, key=lambda c: df[c].notna().sum(), reverse=True)[:n]
    return num_cols

MANIFEST = {}  # we’ll fill this with the assets we generate

In [8]:
# =========================================
# Cell 2 — RQ1: Metrics + OOF diagnostics
# =========================================
def rq1_metrics_tables(metrics_path):
    df = pd.read_csv(metrics_path)
    # normalize column names
    df.columns = [c.strip() for c in df.columns]
    low = {c.lower(): c for c in df.columns}

    # Guess standard columns
    model_col  = low.get("model") or low.get("estimator") or low.get("algorithm") or low.get("run") or df.columns[0]
    metric_col = low.get("metric") or low.get("metric_name") or low.get("name") or None
    value_col  = low.get("value")  or low.get("score")       or low.get("metric_value") or None
    split_col  = low.get("split")  or low.get("dataset")     or None

    if metric_col is None or value_col is None:
        # Attempt wide → long if metrics are columns
        metric_like = [c for c in df.columns if c.lower() not in {model_col.lower(), (split_col or "").lower()} and pd.api.types.is_numeric_dtype(df[c])]
        if metric_like:
            df_long = df.melt(id_vars=[model_col] + ([split_col] if split_col else []),
                              value_vars=metric_like, var_name="metric", value_name="value")
        else:
            raise ValueError("Could not identify metric/value columns in metrics CSV.")
    else:
        df_long = df.rename(columns={model_col: "model", metric_col: "metric", value_col: "value"})
        if split_col and split_col != "split":
            df_long = df_long.rename(columns={split_col: "split"})
    if "model" not in df_long.columns: df_long = df_long.rename(columns={model_col: "model"})

    # Summaries
    pivot_all = df_long.pivot_table(index="model", columns="metric", values="value", aggfunc="mean")
    MANIFEST["rq1_metrics_table"] = save_table(pivot_all.reset_index(), "rq1_metrics_summary", index=False)

    # Per-metric bar charts
    for m in pivot_all.columns:
        fig, ax = plt.subplots()
        pivot_all[m].sort_values(ascending=False).plot(kind="bar", ax=ax)
        ax.set_ylabel(str(m))
        ax.set_title(f"RQ1 • {m} by model")
        assets = save_fig(fig, f"rq1_metric_{str(m).replace(' ', '_')}")
        MANIFEST.setdefault("rq1_metric_charts", []).append(assets)

def rq1_oof_diagnostics(oof_path):
    df = pd.read_csv(oof_path)
    y_col, proba_col, pred_col = pick_label_and_pred_cols(df)
    if y_col is None:
        raise ValueError("Could not identify label/target column in OOF predictions.")
    y = df[y_col].values

    # Classification vs Regression?
    is_class, _ = sniff_binary_col(pd.Series(y))
    task = "classification" if is_class else "regression"

    if task == "classification":
        # pick probability-like column; fallback to pred if it looks 0..1
        p = None
        if proba_col is not None:
            p = df[proba_col].astype(float).values
        elif pred_col is not None and df[pred_col].between(0,1).all():
            p = df[pred_col].astype(float).values

        # Confusion at 0.5 (if we only have binary pred, treat as class)
        if p is None and pred_col is not None:
            yhat = (df[pred_col].astype(float).values > 0.5).astype(int)
        elif p is not None:
            yhat = (p >= 0.5).astype(int)
        else:
            # Try to coerce any numeric pred to 0/1
            candidates = topn_numeric_columns(df, exclude=[y_col], n=1)
            yhat = (df[candidates[0]].astype(float).values >= 0.5).astype(int)

        # Basic confusion matrix
        if sklearn is not None:
            cm = confusion_matrix(y, yhat)
            cm_df = pd.DataFrame(cm, index=["Actual 0","Actual 1"], columns=["Pred 0","Pred 1"])
            MANIFEST["rq1_cm_table"] = save_table(cm_df, "rq1_confusion_matrix", index=True)

        # ROC/PR
        if sklearn is not None and p is not None:
            fpr, tpr, _ = roc_curve(y, p)
            roc_auc = auc(fpr, tpr)
            fig, ax = plt.subplots()
            ax.plot(fpr, tpr, lw=2)
            ax.plot([0,1],[0,1], linestyle="--")
            ax.set_xlabel("FPR")
            ax.set_ylabel("TPR")
            ax.set_title(f"RQ1 • ROC (AUC={roc_auc:.3f})")
            MANIFEST["rq1_roc"] = save_fig(fig, "rq1_roc")

            prec, rec, _ = precision_recall_curve(y, p)
            ap = average_precision_score(y, p)
            fig, ax = plt.subplots()
            ax.plot(rec, prec, lw=2)
            ax.set_xlabel("Recall")
            ax.set_ylabel("Precision")
            ax.set_title(f"RQ1 • Precision–Recall (AP={ap:.3f})")
            MANIFEST["rq1_pr"] = save_fig(fig, "rq1_pr")

            # Calibration / deciles
            df_cal = pd.DataFrame({"y": y, "p": p})
            df_cal["decile"] = pd.qcut(df_cal["p"], 10, labels=False, duplicates="drop")
            cal = df_cal.groupby("decile").agg(avg_p=("p","mean"), rate=("y","mean"), n=("y","size")).reset_index()
            MANIFEST["rq1_calibration_table"] = save_table(cal, "rq1_calibration_by_decile", index=False)

            fig, ax = plt.subplots()
            ax.plot(cal["avg_p"], cal["rate"], marker="o")
            ax.plot([0,1],[0,1], linestyle="--")
            ax.set_xlabel("Average predicted probability")
            ax.set_ylabel("Observed rate")
            ax.set_title("RQ1 • Calibration")
            MANIFEST["rq1_calibration_chart"] = save_fig(fig, "rq1_calibration")
        else:
            print("Note: sklearn not available or probability column not found — skipped ROC/PR/Calibration.")

    else:  # regression
        # pick prediction column
        if pred_col is None:
            # choose the most 'predict-y' numeric column
            candidates = topn_numeric_columns(df, exclude=[y_col])
            if not candidates:
                raise ValueError("No numeric prediction column found for regression.")
            pred_col = candidates[0]
        yhat = df[pred_col].astype(float).values
        # Metrics
        mse = mean_squared_error(y, yhat) if sklearn else np.mean((y-yhat)**2)
        rmse = math.sqrt(mse)
        mae = mean_absolute_error(y, yhat) if sklearn else np.mean(np.abs(y-yhat))
        r2  = r2_score(y, yhat) if sklearn else np.nan
        met = pd.DataFrame([{"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}])
        MANIFEST["rq1_regression_metrics"] = save_table(met, "rq1_regression_metrics", index=False)

        # Parity plot
        fig, ax = plt.subplots()
        ax.scatter(y, yhat, s=10, alpha=0.6)
        mn = min(np.min(y), np.min(yhat)); mx = max(np.max(y), np.max(yhat))
        ax.plot([mn,mx],[mn,mx], linestyle="--")
        ax.set_xlabel("Actual")
        ax.set_ylabel("Predicted")
        ax.set_title("RQ1 • Predicted vs Actual")
        MANIFEST["rq1_parity"] = save_fig(fig, "rq1_parity")

        # Residuals
        res = yhat - y
        fig, ax = plt.subplots()
        ax.hist(res, bins=30)
        ax.set_title("RQ1 • Residuals")
        ax.set_xlabel("Residual")
        MANIFEST["rq1_residuals"] = save_fig(fig, "rq1_residuals")


In [9]:
# =========================================
# Cell 3 — RQ2: Rain quality & monsoon groups
# =========================================
def pick_group_col(df: pd.DataFrame, preferred_keywords=("rain","good","poor","group","monsoon","category","cohort")):
    # prefer columns with keywords and exactly 2–10 unique values
    candidates = []
    for c in df.columns:
        nunq = df[c].nunique(dropna=True)
        if 2 <= nunq <= 10:
            score = sum(k in c.lower() for k in preferred_keywords)
            candidates.append((score, -nunq, c))
    candidates.sort(reverse=True)
    return candidates[0][2] if candidates else None

def rq2_good_vs_poor(gpr_path):
    df = pd.read_csv(gpr_path)
    group = pick_group_col(df)
    if group is None:
        raise ValueError("Could not identify a grouping column (good vs poor rain).")
    # choose top numeric outcomes
    num_cols = topn_numeric_columns(df, exclude=[group], n=12)
    if not num_cols:
        raise ValueError("No numeric outcome columns found in rq2_good_vs_poor_rain.csv")

    # Two-group comparison tables (mean, std, n, diff, pval if scipy)
    summary_rows = []
    groups = [g for g in df[group].dropna().unique()]
    if len(groups) != 2:
        # if >2 groups, keep the top 2 by size
        sizes = df[group].value_counts().index.tolist()[:2]
        groups = sizes

    gA, gB = groups[0], groups[1]
    A = df[df[group]==gA]
    B = df[df[group]==gB]

    for col in num_cols:
        mA, sA, nA = A[col].mean(), A[col].std(), A[col].notna().sum()
        mB, sB, nB = B[col].mean(), B[col].std(), B[col].notna().sum()
        diff = mA - mB
        pval = np.nan
        if scipy is not None:
            try:
                pval = stats.ttest_ind(A[col].dropna(), B[col].dropna(), equal_var=False).pvalue
            except Exception:
                pval = np.nan
        summary_rows.append({"metric": col, f"mean_{gA}": mA, f"mean_{gB}": mB, "diff": diff, "p_value": pval, f"n_{gA}": nA, f"n_{gB}": nB})
    tab = pd.DataFrame(summary_rows).sort_values("diff", ascending=False)
    MANIFEST["rq2_good_poor_table"] = save_table(tab, "rq2_good_vs_poor_summary", index=False)

    # Chart top 8 absolute differences
    top_cols = tab.reindex(tab["diff"].abs().sort_values(ascending=False).index)[:8]["metric"].tolist()
    fig, ax = plt.subplots()
    vals_A = [A[c].mean() for c in top_cols]
    vals_B = [B[c].mean() for c in top_cols]
    x = np.arange(len(top_cols))
    width = 0.4
    ax.bar(x - width/2, vals_A, width, label=str(gA))
    ax.bar(x + width/2, vals_B, width, label=str(gB))
    ax.set_xticks(x, [textwrap.shorten(c, width=30, placeholder="…") for c in top_cols], rotation=20, ha="right")
    ax.legend()
    ax.set_title("RQ2 • Good vs Poor Rain — Top differences")
    MANIFEST["rq2_good_poor_chart"] = save_fig(fig, "rq2_good_vs_poor_topdiffs")

def rq2_monsoon_groups(monsoon_path):
    df = pd.read_csv(monsoon_path)
    group = pick_group_col(df, preferred_keywords=("monsoon","group","cluster","regime","phase","state","category"))
    if group is None:
        raise ValueError("Could not identify monsoon grouping column.")
    num_cols = topn_numeric_columns(df, exclude=[group], n=10)

    # Means by group
    group_means = df.groupby(group)[num_cols].mean().reset_index()
    MANIFEST["rq2_monsoon_table"] = save_table(group_means, "rq2_monsoon_group_means", index=False)

    # Chart: for each top metric, show grouped bars (limit to 5 metrics to keep chart clean)
    for col in num_cols[:5]:
        fig, ax = plt.subplots()
        ax.bar(group_means[group].astype(str), group_means[col].values)
        ax.set_title(f"RQ2 • {col} by {group}")
        ax.set_xlabel(group)
        ax.set_ylabel(col)
        MANIFEST.setdefault("rq2_monsoon_charts", []).append(save_fig(fig, f"rq2_monsoon_{col}"))


In [10]:
# =========================================
# Cell 4 — RQ3: GDP interactions/enrichment
# =========================================
def rq3_gdp_enrichment(enriched_path):
    df = pd.read_csv(enriched_path)
    cols_low = {c.lower(): c for c in df.columns}
    # Try to find GDP and outcome-ish columns
    gdp_pred = next((cols_low[k] for k in ["gdp_pred","pred_gdp","gdp_prediction","gdp_hat"] if k in cols_low), None)
    gdp      = next((cols_low[k] for k in ["gdp","gdp_actual","gdp_true"] if k in cols_low), None)

    outcome_cands = ["y","target","label","outcome","response"]
    outcome = next((cols_low[k] for k in outcome_cands if k in cols_low), None)

    # If not found, just take top numeric columns as outcomes (excluding GDP fields)
    exclude = {c for c in [gdp, gdp_pred] if c}
    outcomes = [outcome] if outcome else topn_numeric_columns(df, exclude=exclude, n=5)

    # Correlation table (Pearson)
    rows = []
    base_vars = [c for c in [gdp, gdp_pred] if c] or []
    for base in base_vars:
        for tgt in outcomes:
            if base == tgt: 
                continue
            series_base = pd.to_numeric(df[base], errors="coerce")
            series_tgt  = pd.to_numeric(df[tgt],  errors="coerce")
            valid = series_base.notna() & series_tgt.notna()
            if valid.sum() < 3:
                r = np.nan; p = np.nan
            else:
                if scipy is not None:
                    r, p = stats.pearsonr(series_base[valid], series_tgt[valid])
                else:
                    r = np.corrcoef(series_base[valid], series_tgt[valid])[0,1]
                    p = np.nan
            rows.append({"x": base, "y": tgt, "pearson_r": r, "p_value": p, "n": int(valid.sum())})
    corr_tab = pd.DataFrame(rows).sort_values("pearson_r", ascending=False)
    MANIFEST["rq3_corr_table"] = save_table(corr_tab, "rq3_gdp_correlations", index=False)

    # Scatter charts with simple linear fit, for top 4 |r|
    top_pairs = corr_tab.reindex(corr_tab["pearson_r"].abs().sort_values(ascending=False).index)[:4]
    for _, row in top_pairs.iterrows():
        xcol, ycol = row["x"], row["y"]
        x = pd.to_numeric(df[xcol], errors="coerce")
        y = pd.to_numeric(df[ycol], errors="coerce")
        m = ~ (x.isna() | y.isna())
        if m.sum() < 3: 
            continue
        # simple fit
        coef = np.polyfit(x[m], y[m], 1)
        fx = np.poly1d(coef)
        fig, ax = plt.subplots()
        ax.scatter(x[m], y[m], s=12, alpha=0.6)
        xx = np.linspace(x[m].min(), x[m].max(), 100)
        ax.plot(xx, fx(xx), lw=2)
        ax.set_xlabel(xcol); ax.set_ylabel(ycol)
        ax.set_title(f"RQ3 • {xcol} vs {ycol} (r={row['pearson_r']:.2f})")
        MANIFEST.setdefault("rq3_scatter_charts", []).append(save_fig(fig, f"rq3_scatter_{xcol}_vs_{ycol}".replace(" ","_")))


In [11]:
# =========================================
# Cell 5 — RQ4: Importance & Interaction Uplift
# =========================================
def rq4_permutation_importance(perm_path):
    df = pd.read_csv(perm_path)
    # Try standard columns
    cols_low = {c.lower(): c for c in df.columns}
    feat = cols_low.get("feature") or cols_low.get("variable") or df.columns[0]
    imp  = cols_low.get("importance") or cols_low.get("permutation_importance") or cols_low.get("score") or df.columns[1]

    # Aggregated (if multiple seeds/folds)
    imp_tbl = df.groupby(feat)[imp].agg(["mean","std","count"]).reset_index().sort_values("mean", ascending=False)
    MANIFEST["rq4_perm_table"] = save_table(imp_tbl, "rq4_permutation_importance", index=False)

    # Bar chart top 20
    top = imp_tbl.head(20)
    fig, ax = plt.subplots(figsize=(8, max(3.5, 0.35*len(top))))
    ax.barh(top[feat].astype(str)[::-1], top["mean"][::-1].values)
    ax.set_xlabel("Permutation importance (mean)")
    ax.set_title("RQ4 • Top features by permutation importance")
    MANIFEST["rq4_perm_chart"] = save_fig(fig, "rq4_permutation_importance_top20")

def rq4_interaction_uplift(uplift_path):
    df = pd.read_csv(uplift_path)
    cols_low = {c.lower(): c for c in df.columns}
    # Heuristics
    feat = cols_low.get("feature") or cols_low.get("variable") or None
    uplift = cols_low.get("uplift") or cols_low.get("tau") or cols_low.get("effect") or None
    decile = cols_low.get("decile") or cols_low.get("bucket") or cols_low.get("quantile") or None
    treat  = cols_low.get("treatment") or cols_low.get("treat") or None

    # If we have decile uplift, chart the curve
    if decile and uplift:
        curve = df.groupby(decile)[uplift].mean().reset_index()
        fig, ax = plt.subplots()
        ax.plot(curve[decile], curve[uplift], marker="o")
        ax.set_title("RQ4 • Uplift by decile/bucket")
        ax.set_xlabel(decile); ax.set_ylabel(uplift)
        MANIFEST["rq4_uplift_curve"] = save_fig(fig, "rq4_uplift_by_decile")

    # If we have per-feature uplift effects, make a table/chart
    if feat and uplift:
        eff = df.groupby(feat)[uplift].agg(["mean","std","count"]).reset_index().sort_values("mean", ascending=False)
        MANIFEST["rq4_uplift_table"] = save_table(eff, "rq4_feature_uplift_summary", index=False)

        fig, ax = plt.subplots(figsize=(8, max(3.5, 0.35*min(25, len(eff)))))
        top = eff.head(25)
        ax.barh(top[feat].astype(str)[::-1], top["mean"][::-1].values)
        ax.set_xlabel("Estimated uplift (mean)")
        ax.set_title("RQ4 • Top features by interaction uplift")
        MANIFEST["rq4_uplift_chart"] = save_fig(fig, "rq4_top_feature_uplifts")


In [12]:
# =========================================
# Cell 6 — Orchestration: find files & run
# =========================================
FILE_HINTS = {
    "rq1_metrics": [
        "rq1_metrics.csv"
    ],
    "rq1_oof": [
        "rq1_oof_predictions.csv",   # dev upload name
        "rq1_oofpredictions.csv"     # user's earlier variant
    ],
    "rq2_good_poor": [
        "rq2_good_vs_poor_rain.csv"
    ],
    "rq2_monsoon": [
        "rq2_monsoon_groups.csv"
    ],
    "rq3_enriched": [
        "rq3_enriched_with_gdp_pred.csv"
    ],
    "rq4_uplift": [
        "rq4_interaction_uplift.csv"
    ],
    "rq4_perm": [
        "rq4_permutation_importance.csv",
        "permutation_importance_holdout.csv",
        "permutation_importance.csv"
    ],
}

def run_all(verbose=True):
    resolved = {k: find_file(v) for k, v in FILE_HINTS.items()}
    if verbose:
        print("Resolved file paths:")
        for k, p in resolved.items():
            print(f"  {k}: {p if p else 'NOT FOUND'}")
        print()

    # RQ1
    if resolved["rq1_metrics"]:
        rq1_metrics_tables(resolved["rq1_metrics"])
    else:
        print("Skipped RQ1 metrics: file not found.")

    if resolved["rq1_oof"]:
        rq1_oof_diagnostics(resolved["rq1_oof"])
    else:
        print("Skipped RQ1 OOF diagnostics: file not found.")

    # RQ2
    if resolved["rq2_good_poor"]:
        rq2_good_vs_poor(resolved["rq2_good_poor"])
    else:
        print("Skipped RQ2 good vs poor rain: file not found.")
    if resolved["rq2_monsoon"]:
        rq2_monsoon_groups(resolved["rq2_monsoon"])
    else:
        print("Skipped RQ2 monsoon groups: file not found.")

    # RQ3
    if resolved["rq3_enriched"]:
        rq3_gdp_enrichment(resolved["rq3_enriched"])
    else:
        print("Skipped RQ3 GDP enrichment: file not found.")

    # RQ4
    if resolved["rq4_perm"]:
        rq4_permutation_importance(resolved["rq4_perm"])
    else:
        print("Skipped RQ4 permutation importance: file not found.")
    if resolved["rq4_uplift"]:
        rq4_interaction_uplift(resolved["rq4_uplift"])
    else:
        print("Skipped RQ4 interaction uplift: file not found.")

    # Save manifest of assets for easy embedding
    manifest_path = OUT_ROOT / "manifest.json"
    with open(manifest_path, "w") as f:
        json.dump(MANIFEST, f, indent=2)
    if verbose:
        print(f"\nWrote asset manifest: {manifest_path.resolve()}")
        print(f"Tables → {OUT_TABLES.resolve()}")
        print(f"Charts → {OUT_CHARTS.resolve()}")

# Convenience: run immediately if you like
# run_all()


In [13]:
# =========================================
# Cell 7 — Run everything
# =========================================
run_all(verbose=True)


Resolved file paths:
  rq1_metrics: NOT FOUND
  rq1_oof: NOT FOUND
  rq2_good_poor: NOT FOUND
  rq2_monsoon: NOT FOUND
  rq3_enriched: NOT FOUND
  rq4_uplift: NOT FOUND
  rq4_perm: processed\permutation_importance_holdout.csv

Skipped RQ1 metrics: file not found.
Skipped RQ1 OOF diagnostics: file not found.
Skipped RQ2 good vs poor rain: file not found.
Skipped RQ2 monsoon groups: file not found.
Skipped RQ3 GDP enrichment: file not found.
Skipped RQ4 interaction uplift: file not found.

Wrote asset manifest: C:\Users\Local User\Documents\GitHub\marketpredict\report_assets\manifest.json
Tables → C:\Users\Local User\Documents\GitHub\marketpredict\report_assets\tables
Charts → C:\Users\Local User\Documents\GitHub\marketpredict\report_assets\charts
